# DNAmPhenoAge

Let's first import some packages:

In [1]:
import os
import inspect
import shutil
import json
import math
import torch
import pandas as pd
import pyaging as pya

/work/gr-fe/bryan/miniconda3/envs/pyaging/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.DNAmPhenoAge)

class DNAmPhenoAge(pyagingModel):
    def __init__(self):
        super().__init__()

    def preprocess(self, x):
        if self.reference_values is None:
            return x
        if isinstance(self.reference_values, torch.Tensor):
            reference = self.reference_values.to(device=x.device, dtype=x.dtype)
        else:
            reference = torch.tensor(self.reference_values, device=x.device, dtype=x.dtype)
        return torch.where(torch.isnan(x), reference, x)

    def postprocess(self, x):
        return x



In [3]:
model = pya.models.DNAmPhenoAge()

## Define clock metadata

In [25]:
model.metadata["clock_name"] = "PhenoAge"
model.metadata["data_type"] = "DNA methylation"  # Paper: methylation
model.metadata["species"] = "Homo sapiens"  # Paper: Homo sapiens
model.metadata["year"] = 2018
model.metadata["approved_by_author"] = "⌛"
model.metadata["citation"] = "Levine, M. E., et al. \"An epigenetic biomarker of aging for lifespan and healthspan.\" Aging 10.4 (2018): 573-591."
model.metadata["doi"] = "https://doi.org/10.18632/aging.101414"
model.metadata["notes"] = 'Blood DNA-methylation estimator trained by elastic net on 513 CpGs common to the 27K, 450K and EPIC arrays to reproduce a mortality-derived clinical Phenotypic Age. The parent DNAmPhenoAge coefficients were fitted on InCHIANTI 450K whole-blood data; the selected sites also occur on 27K and EPIC arrays.'
model.metadata["research_only"] = None
model.metadata["tissue"] = ["whole blood"]  # Paper: whole blood
model.metadata["predicts"] = ["phenotypic age"]  # Paper: phenotypic age
model.metadata["training_target"] = ["phenotypic age"]  # Paper: phenotypic age
model.metadata["unit"] = ["years"]  # Paper: years
model.metadata["model_type"] = "elastic net regression"  # Paper: Elastic net
model.metadata["platform"] = ['Illumina 450K']  # Paper: The parent DNAmPhenoAge model was fitted in InCHIANTI using NOOB-normalized 450K whole-blood methylation. Common 27K/450K/EPIC sites were chosen for compatibility; JHS EPIC data were validation.
model.metadata["population"] = "adults"  # Paper: adults (21-100 years)
model.metadata["journal"] = "Aging"
model.metadata["last_author"] = "Steve Horvath"
model.metadata["n_features"] = 513
model.metadata["citations"] = 3848
model.metadata["citations_date"] = '2026-10-02'


## Download clock dependencies

#### Download directly with curl

In [26]:
### Download paper weights

# https://pmc.ncbi.nlm.nih.gov/articles/instance/5940111/bin/aging-10-101414-s002.csv

## Load features

#### From CSV file

In [29]:
df = pd.read_csv('pyaging_model/phenoage.csv')
model.features = df['CpG'][1:].tolist()

## Load weights into base model

In [30]:
weights = torch.tensor(df['Weight'][1:].tolist()).unsqueeze(0).float()
intercept = torch.tensor([df['Weight'][0]]).float()

#### Linear model

In [31]:
base_model = pya.models.LinearModel(input_dim=len(model.features))

base_model.linear.weight.data = weights.float()
base_model.linear.bias.data = intercept.float()

model.base_model = base_model

## Load reference values

In [32]:
model.reference_values = None

## Load preprocess and postprocess objects

In [33]:
model.preprocess_name = None
model.preprocess_dependencies = None

In [34]:
model.postprocess_name = None
model.postprocess_dependencies = None

## Check all clock parameters

In [35]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: False
metadata: {'approved_by_author': '⌛',
 'citation': 'Levine, M. E., et al. "An epigenetic biomarker of aging for '
             'lifespan and healthspan." Aging 10.4 (2018): 573-591.',
 'citations': 3848,
 'citations_date': '2026-10-02',
 'clock_name': 'PhenoAge',
 'data_type': 'DNA methylation',
 'doi': 'https://doi.org/10.18632/aging.101414',
 'journal': 'Aging',
 'last_author': 'Steve Horvath',
 'model_type': 'elastic net regression',
 'n_features': 513,
 'notes': 'Blood DNA-methylation estimator trained by elastic net on 513 CpGs '
          'common to the 27K, 450K and EPIC arrays to reproduce a '
          'mortality-derived clinical Phenotypic Age. The parent DNAmPhenoAge '
          'coefficients were fitted on InCHIANTI 450K whole-blood data; the '
          'selected sites also occur on 27K and EPIC arrays.',
 'platform': ['Illumina 450K'],
 'population

## Normal feature ranges

In [36]:
# Units and plausibility ranges come from the package registry, keyed by feature name.
feature_ranges = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
model.feature_units = [record["unit"] for record in feature_ranges]
pd.DataFrame.from_records(feature_ranges).head()

,feature,unit,low,high
0,cg15611364,beta value,0.0,1.0
1,cg17605084,beta value,0.0,1.0
2,cg26382071,beta value,0.0,1.0
3,cg12743894,beta value,0.0,1.0
4,cg19287114,beta value,0.0,1.0


## Basic test

In [37]:
# Exercise the clock with values in the middle of each feature's expected range.
records = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
midpoints = [
    (record["low"] + record["high"]) / 2 if math.isfinite(record["high"]) else max(record["low"], 1.0)
    for record in records
]
input = torch.tensor([midpoints] * 10, dtype=torch.float64)
model.eval()
model.to(torch.float64)
pred = model(input)
pred

tensor([[51.2400],
        [51.2400],
        [51.2400],
        [51.2400],
        [51.2400],
        [51.2400],
        [51.2400],
        [51.2400],
        [51.2400],
        [51.2400]], dtype=torch.float64, grad_fn=<AddmmBackward0>)

## Save torch model

In [38]:
torch.save(model, f"pyaging_model/{model.metadata['clock_name']}.pt")